# P1 vs P2 Comparison Diagnostics

This notebook isolates the MODIS Terra-only vs Terra+Aqua transition:

- **P1**: 2000-06-01 to 2002-06-30, MODIS Terra SCF
- **P2**: 2002-07-01 to 2007-05-31, MODIS Terra+Aqua SCF

The comparison uses **common support** wherever possible. For gridded products, cells must be valid for P1 and P2 and for OL and DA. For station products, stations must meet the minimum sample threshold in both periods and both experiments.

Outputs are written to `projects/M21C_ls/output/p1_p2_comparison/`.

In [ ]:
from pathlib import Path
import os
import sys

os.environ.setdefault("MKL_THREADING_LAYER", "SEQUENTIAL")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

import numpy as np
import pandas as pd
import xarray as xr

try:
    from IPython import get_ipython
    from IPython.display import display
except Exception:
    def get_ipython():
        return None
    def display(obj):
        print(obj)


def running_in_notebook() -> bool:
    shell = get_ipython()
    return shell is not None and shell.__class__.__name__ == "ZMQInteractiveShell"


IN_NOTEBOOK = running_in_notebook()

import matplotlib
if not IN_NOTEBOOK:
    matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors


def show_figure(fig):
    """Show figures inline in notebooks; avoid GUI windows in batch/headless runs."""
    if IN_NOTEBOOK:
        display(fig)
    plt.close(fig)

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except Exception as exc:
    HAS_CARTOPY = False
    print("Cartopy unavailable; map cells will fall back or should be skipped:", exc)


def find_repo_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / ".git").exists() and (p / "common/python/io/read_GEOSldas.py").exists():
            return p
    raise FileNotFoundError("Could not locate geosldas-analysis repo root")


HERE = Path.cwd().resolve()
REPO_ROOT = find_repo_root(HERE)
PROJECT_ROOT = REPO_ROOT / "projects/M21C_ls"
OUT_DIR = PROJECT_ROOT / "output/p1_p2_comparison"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

GEOSLDAS_DIAG_ROOT = Path("/Users/amfox/Desktop/GEOSldas_diagnostics/test_data/M21C_land_sweeper_v2")
EASE_PATH = REPO_ROOT / "common/python/plotting/ease_grids"

print("REPO_ROOT:", REPO_ROOT)
print("OUT_DIR:", OUT_DIR)
print("GEOSLDAS_DIAG_ROOT exists:", GEOSLDAS_DIAG_ROOT.exists())

In [ ]:
# Runtime switches. ERA5-Land and GHCN are local but heavier than OFA/IMS/SNOTEL.
RUN_OFA = True
RUN_IMS = True
RUN_SNOTEL = True
RUN_GHCN = True
RUN_ERA5L = True   # reads the 2 x 7.9 GB aligned monthly files; set False for a quick run
MAKE_MAPS = True

P1P2_PERIODS = pd.DataFrame([
    {"period_id": "P1", "start": "2000-06-01", "end": "2002-06-30", "label": "MODIS Terra SCF"},
    {"period_id": "P2", "start": "2002-07-01", "end": "2007-05-31", "label": "MODIS Terra+Aqua SCF"},
])
P1P2_PERIODS["start"] = pd.to_datetime(P1P2_PERIODS["start"])
P1P2_PERIODS["end"] = pd.to_datetime(P1P2_PERIODS["end"])
P1P2_PERIODS["n_days_inclusive"] = (P1P2_PERIODS["end"] - P1P2_PERIODS["start"]).dt.days + 1
P1P2_PERIODS["n_months"] = P1P2_PERIODS.apply(lambda r: len(pd.period_range(r["start"], r["end"], freq="M")), axis=1)
P1P2_BY_ID = P1P2_PERIODS.set_index("period_id")

PERIOD_FILE_DATES = {
    "P1": ("20000601", "20020630"),
    "P2": ("20020701", "20070531"),
}

COMMON_SUPPORT = {
    "station_min_common_days": 100,
    "era5l_min_common_months": {"P1": 18, "P2": 36},
    "ofa_min_species_obs": 20,
}

display(P1P2_PERIODS)

In [ ]:
def rel(path: str) -> Path:
    return REPO_ROOT / path


def diag(path: str) -> Path:
    return GEOSLDAS_DIAG_ROOT / path


INPUTS = {
    "ofa_p1_ol": diag("temporal_stats_OL_20000601_20020630.nc4"),
    "ofa_p1_da": diag("temporal_stats_DA_20000601_20020630.nc4"),
    "ofa_p2_ol": diag("temporal_stats_OL_20020701_20070531.nc4"),
    "ofa_p2_da": diag("temporal_stats_DA_20020701_20070531.nc4"),
    "tilecoord": diag("LS_OLv8_M36.ldas_tilecoord.bin"),
    "ims_counts": rel("projects/IMS/output/ims_ol_da_cell_counts_metrics_SMAP_EASEv2_M36_GLOBAL_2000_2007_thr0p50_imsSnowDaysGe10_terraAquaScopes.nc4"),
    "ims_table": rel("projects/IMS/output/ims_ol_da_comparison_table_SMAP_EASEv2_M36_GLOBAL_2000_2007_thr0p50_imsSnowDaysGe10_terraAquaScopes.csv"),
    "snotel_raw": rel("projects/SNOTEL/outputs_snotel_ol_da_validation/snotel_raw_timeseries_SMAP_EASEv2_M36_GLOBAL_20000601_20240601.nc"),
    "ghcn_raw": rel("projects/GHCN_snwd/outputs_ghcn_snwd_ol_da_validation/ghcn_snwd_raw_timeseries_SMAP_EASEv2_M36_GLOBAL_20000101_20241231.nc"),
    "era5l_ol": rel("projects/era5_land/notebooks/ERA5L_vs_OLv8_M36_strict_summary.nc"),
    "era5l_da": rel("projects/era5_land/notebooks/ERA5L_vs_DAv8_M36_strict_summary.nc"),
}

availability = []
for key, path in INPUTS.items():
    availability.append({
        "key": key,
        "path": str(path),
        "exists": path.exists(),
        "size_gb": path.stat().st_size / 1e9 if path.exists() else np.nan,
    })
availability_df = pd.DataFrame(availability)
availability_df.to_csv(OUT_DIR / "input_availability.csv", index=False)
display(availability_df)

## Shared Metrics and Plot Helpers

In [ ]:
def safe_corr_1d(obs: np.ndarray, mod: np.ndarray) -> float:
    obs = np.asarray(obs, dtype=float)
    mod = np.asarray(mod, dtype=float)
    valid = np.isfinite(obs) & np.isfinite(mod)
    if valid.sum() < 2:
        return np.nan
    x = obs[valid]
    y = mod[valid]
    sx = np.std(x)
    sy = np.std(y)
    if sx <= 0 or sy <= 0:
        return np.nan
    return float(np.corrcoef(x, y)[0, 1])


def vector_metrics(obs, mod, dim="time", min_pairs=1):
    valid = np.isfinite(obs) & np.isfinite(mod)
    n = valid.sum(dim)
    diff = (mod - obs).where(valid)
    bias = diff.mean(dim, skipna=True)
    centered = diff - bias
    rmse = np.sqrt((diff ** 2).mean(dim, skipna=True))
    ubrmse = np.sqrt((centered ** 2).mean(dim, skipna=True))
    obs_mean = obs.where(valid).mean(dim, skipna=True)
    mod_mean = mod.where(valid).mean(dim, skipna=True)
    obs_anom = obs.where(valid) - obs_mean
    mod_anom = mod.where(valid) - mod_mean
    cov = (obs_anom * mod_anom).sum(dim, skipna=True)
    denom = np.sqrt((obs_anom ** 2).sum(dim, skipna=True) * (mod_anom ** 2).sum(dim, skipna=True))
    corr = xr.where(denom > 0, cov / denom, np.nan)
    out = xr.Dataset({"N": n, "R": corr, "bias": bias, "abs_bias": np.abs(bias), "rmse": rmse, "ubrmse": ubrmse})
    return out.where(n >= min_pairs)


def improvement_delta(metric: str, da, ol):
    if metric in {"R", "anomR", "accuracy", "hit_rate", "correct_rejection_rate"}:
        return da - ol
    if metric in {"rmse", "ubrmse", "abs_bias", "miss_rate", "false_alarm_ratio"}:
        return ol - da
    return da - ol


def panel_label(ax, label, x=-0.075, y=1.04):
    ax.text(x, y, label, transform=ax.transAxes, ha="left", va="bottom", fontsize=10, fontweight="bold")


def add_geo_base(ax, extent=(-180, 180, -60, 90)):
    if not HAS_CARTOPY:
        return
    ax.add_feature(cfeature.LAND, facecolor="0.92", edgecolor="none", zorder=0)
    ax.coastlines(resolution="50m", linewidth=0.45, color="0.35")
    ax.set_extent(extent, crs=ccrs.PlateCarree())


def savefig(fig, stem: str):
    png = FIG_DIR / f"{stem}.png"
    pdf = FIG_DIR / f"{stem}.pdf"
    fig.savefig(png, dpi=300, bbox_inches="tight")
    fig.savefig(pdf, bbox_inches="tight")
    print("saved", png)
    print("saved", pdf)
    return png, pdf

## OFA Counts and MODIS OmF

Uses the P1/P2 period-split temporal-stat files generated from the monthly ObsFcstAna sums. Common tile support is applied across P1, P2, OL, and DA for MODIS.

In [ ]:
if RUN_OFA:
    from netCDF4 import Dataset
    sys.path.append(str(REPO_ROOT / "common/python/io"))
    from read_GEOSldas import read_tilecoord

    SPECIES_GROUPS = {
        "SMOS": [0, 1, 2, 3],
        "SMAP": [4, 5, 6, 7],
        "ASCAT": [8, 9, 10],
        "CYGN": [13],
        "MODIS": [11, 12],
    }
    OFA_GROUPS = ["MODIS"]
    NMIN = COMMON_SUPPORT["ofa_min_species_obs"]

    def load_temporal_stats(path: Path) -> dict[str, np.ndarray]:
        stats = {}
        with Dataset(path, "r") as nc:
            for key, value in nc.variables.items():
                arr = value[:]
                if hasattr(arr, "filled"):
                    arr = arr.filled(np.nan)
                stats[key] = np.asarray(arr, dtype=float)
        return stats

    def weighted_group(stats, group):
        idx = SPECIES_GROUPS[group]
        n_raw = stats["N_data"][:, idx].copy()
        weights = n_raw.copy()
        weights[weights < NMIN] = 0
        group_n = np.nansum(weights, axis=1)
        out = {"Nobs_data": np.nansum(n_raw, axis=1), "Nobs_weighted": group_n}
        for key in ["OmF_stdv", "OmF_norm_stdv", "OmF_mean", "OmA_stdv"]:
            arr = stats[key][:, idx].copy()
            arr[n_raw < NMIN] = np.nan
            out[key] = np.divide(
                np.nansum(arr * weights, axis=1),
                group_n,
                out=np.full(group_n.shape, np.nan, dtype=float),
                where=group_n > 0,
            )
        return out

    ofa = {}
    for pid in ["P1", "P2"]:
        start_s, end_s = PERIOD_FILE_DATES[pid]
        ofa[(pid, "OL")] = {g: weighted_group(load_temporal_stats(diag(f"temporal_stats_OL_{start_s}_{end_s}.nc4")), g) for g in OFA_GROUPS}
        ofa[(pid, "DA")] = {g: weighted_group(load_temporal_stats(diag(f"temporal_stats_DA_{start_s}_{end_s}.nc4")), g) for g in OFA_GROUPS}

    ofa_rows = []
    ofa_map_values = {}
    for group in OFA_GROUPS:
        common = np.ones_like(ofa[("P1", "OL")][group]["Nobs_data"], dtype=bool)
        for pid in ["P1", "P2"]:
            for exp in ["OL", "DA"]:
                common &= ofa[(pid, exp)][group]["Nobs_weighted"] > 0
                common &= np.isfinite(ofa[(pid, exp)][group]["OmF_stdv"])
        common &= ofa[("P1", "OL")][group]["OmF_stdv"] > 0.001
        common &= ofa[("P2", "OL")][group]["OmF_stdv"] > 0.001

        for pid in ["P1", "P2"]:
            p = P1P2_BY_ID.loc[pid]
            ol = ofa[(pid, "OL")][group]
            da = ofa[(pid, "DA")][group]
            improvement = np.divide(
                ol["OmF_stdv"] - da["OmF_stdv"],
                ol["OmF_stdv"],
                out=np.full_like(ol["OmF_stdv"], np.nan),
                where=ol["OmF_stdv"] > 0,
            ) * 100.0
            ofa_map_values[(pid, group, "obs_per_day")] = da["Nobs_data"] / p["n_days_inclusive"]
            ofa_map_values[(pid, group, "omf_improvement_percent")] = np.where(common, improvement, np.nan)
            ofa_map_values[(pid, group, "da_omf_stdv")] = np.where(common, da["OmF_stdv"], np.nan)

            for exp, vals in [("OL", ol), ("DA", da)]:
                total = float(np.nansum(vals["Nobs_data"][common]))
                ntiles = int(common.sum())
                ofa_rows.extend([
                    {"period_id": pid, "period_label": p["label"], "group": group, "experiment": exp, "metric": "total_obs_common_tiles", "value": total, "unit": "observations", "n_tiles_common": ntiles},
                    {"period_id": pid, "period_label": p["label"], "group": group, "experiment": exp, "metric": "obs_per_day_common_tiles", "value": total / p["n_days_inclusive"], "unit": "observations/day", "n_tiles_common": ntiles},
                    {"period_id": pid, "period_label": p["label"], "group": group, "experiment": exp, "metric": "spatial_mean_OmF_stdv_common_tiles", "value": float(np.nanmean(vals["OmF_stdv"][common])), "unit": "native", "n_tiles_common": ntiles},
                ])
            ofa_rows.append({"period_id": pid, "period_label": p["label"], "group": group, "experiment": "DA_vs_OL", "metric": "spatial_mean_OmF_stdv_improvement_common_tiles", "value": float(np.nanmean(improvement[common])), "unit": "percent", "n_tiles_common": int(common.sum())})

    ofa_summary = pd.DataFrame(ofa_rows)
    ofa_summary.to_csv(OUT_DIR / "p1_p2_ofa_summary.csv", index=False)
    display(ofa_summary)

In [ ]:
if RUN_OFA and MAKE_MAPS:
    tilecoord = read_tilecoord(str(INPUTS["tilecoord"]))
    lats2d = np.fromfile(str(EASE_PATH / "EASE2_M36km.lats.964x406x1.double"), dtype=np.float64).reshape((406, 964))
    lons2d = np.fromfile(str(EASE_PATH / "EASE2_M36km.lons.964x406x1.double"), dtype=np.float64).reshape((406, 964))
    tile_rows = np.asarray(tilecoord["j_indg"], dtype=int)
    tile_cols = np.asarray(tilecoord["i_indg"], dtype=int)

    def grid_from_tile_values(values, mask_antarctica=True):
        grid = np.full(lats2d.shape, np.nan, dtype=float)
        vals = np.asarray(values, dtype=float)
        valid = np.isfinite(vals)
        grid[tile_rows[valid], tile_cols[valid]] = vals[valid]
        if mask_antarctica:
            grid = np.where(lats2d < -60, np.nan, grid)
        return grid

    map_specs = [
        ("obs_per_day", "DA obs/day", "viridis", None, "observations/day"),
        ("omf_improvement_percent", "OmF StdDev improvement", "RdBu_r", mcolors.TwoSlopeNorm(vmin=-30, vcenter=0, vmax=30), "% (red = DA better)"),
        ("da_omf_stdv", "DA OmF StdDev", "magma", None, "native"),
    ]
    fig = plt.figure(figsize=(12.5, 5.4))
    projection = ccrs.Robinson() if HAS_CARTOPY else None
    axes = []
    ims = []
    for r, pid in enumerate(["P1", "P2"]):
        for c, (key, title, cmap, norm, cbar_label) in enumerate(map_specs):
            ax = fig.add_subplot(2, 3, r * 3 + c + 1, projection=projection) if HAS_CARTOPY else fig.add_subplot(2, 3, r * 3 + c + 1)
            axes.append(ax)
            arr = grid_from_tile_values(ofa_map_values[(pid, "MODIS", key)])
            if HAS_CARTOPY:
                add_geo_base(ax)
                im = ax.pcolormesh(lons2d, lats2d, arr, transform=ccrs.PlateCarree(), cmap=cmap, norm=norm, shading="auto", rasterized=True)
            else:
                im = ax.imshow(arr, cmap=cmap, norm=norm, origin="lower")
            ims.append((im, c, cbar_label))
            if r == 0:
                ax.set_title(title, fontsize=10)
            if c == 0:
                ax.text(-0.07, 0.5, f"{pid}\n{P1P2_BY_ID.loc[pid, 'label']}", transform=ax.transAxes, ha="right", va="center", fontsize=9, fontweight="bold")
            panel_label(ax, f"({chr(97 + r * 3 + c)})")
    for c in range(3):
        cims = [im for im, col, _ in ims if col == c]
        label = [lab for _, col, lab in ims if col == c][0]
        cbar = fig.colorbar(cims[0], ax=[axes[c], axes[c + 3]], orientation="horizontal", fraction=0.05, pad=0.05)
        cbar.set_label(label, fontsize=8)
    fig.suptitle("MODIS OFA P1 vs P2 diagnostics on common tile support", fontsize=12)
    savefig(fig, "p1_p2_ofa_modis_maps")
    show_figure(fig)

## IMS Snow-Cover Skill

Uses the Terra/Terra+Aqua custom IMS scope product. The maps show DA improvement relative to OL. Positive/red means DA is better for every metric.

In [ ]:
if RUN_IMS:
    ims_metric_specs = [
        {"key": "accuracy", "label": "Accuracy", "higher_better": True},
        {"key": "hit_rate", "label": "Hit rate", "higher_better": True},
        {"key": "miss_rate", "label": "Miss rate", "higher_better": False},
        {"key": "false_alarm_ratio", "label": "False alarm ratio", "higher_better": False},
        {"key": "correct_rejection_rate", "label": "Correct rejection rate", "higher_better": True},
    ]
    ims_scope_for_period = {"P1": "P1_MODIS_Terra_SCF", "P2": "P2_MODIS_Terra_Aqua_SCF"}
    ims_summary_rows = []

    with xr.open_dataset(INPUTS["ims_counts"]) as ds:
        scope_names = np.asarray(ds["scope_name"].values).astype(str)
        scope_idx = {pid: int(np.where(scope_names == name)[0][0]) for pid, name in ims_scope_for_period.items()}
        lon = np.asarray(ds["cell_lon"].values, dtype=float)
        lat = np.asarray(ds["cell_lat"].values, dtype=float)
        eligible = np.asarray(ds["cell_eligible"].values, dtype=bool)
        ims_map = {}
        for pid in ["P1", "P2"]:
            idx = scope_idx[pid]
            n_common = np.minimum(np.asarray(ds["N"].isel(experiment=0, scope=idx).values, dtype=float), np.asarray(ds["N"].isel(experiment=1, scope=idx).values, dtype=float))
            domain = eligible & np.isfinite(lon) & np.isfinite(lat) & (lat >= 0)
            for spec in ims_metric_specs:
                key = spec["key"]
                sign = 1.0 if spec["higher_better"] else -1.0
                direction = "DA - OL" if spec["higher_better"] else "OL - DA"
                ol = np.asarray(ds[key].isel(experiment=0, scope=idx).values, dtype=float)
                da = np.asarray(ds[key].isel(experiment=1, scope=idx).values, dtype=float)
                improvement = sign * (da - ol)
                valid = domain & np.isfinite(improvement)
                ims_map[(pid, key)] = improvement
                ims_summary_rows.append({
                    "period_id": pid,
                    "period_label": P1P2_BY_ID.loc[pid, "label"],
                    "metric": key,
                    "metric_label": spec["label"],
                    "improvement_definition": direction,
                    "n_cells": int(valid.sum()),
                    "n_pairs": int(np.nansum(n_common[valid])),
                    "mean_improvement": float(np.nanmean(improvement[valid])),
                    "median_improvement": float(np.nanmedian(improvement[valid])),
                    "percent_cells_improved": float(np.nanmean(improvement[valid] > 0) * 100.0),
                })
    ims_summary = pd.DataFrame(ims_summary_rows)
    ims_summary.to_csv(OUT_DIR / "p1_p2_ims_summary.csv", index=False)
    display(ims_summary)

In [ ]:
if RUN_IMS and MAKE_MAPS:
    fig = plt.figure(figsize=(15, 4.9))
    projection = ccrs.Robinson() if HAS_CARTOPY else None
    cmap = plt.get_cmap("RdBu_r").copy()
    cmap.set_bad("#d4d4d4")
    norm = mcolors.TwoSlopeNorm(vmin=-0.25, vcenter=0, vmax=0.25)
    axes = []
    scatter = None
    with xr.open_dataset(INPUTS["ims_counts"]) as ds:
        lon = np.asarray(ds["cell_lon"].values, dtype=float)
        lat = np.asarray(ds["cell_lat"].values, dtype=float)
        eligible = np.asarray(ds["cell_eligible"].values, dtype=bool)
        domain = eligible & np.isfinite(lon) & np.isfinite(lat) & (lat >= 0)
        for r, pid in enumerate(["P1", "P2"]):
            for c, spec in enumerate(ims_metric_specs):
                ax = fig.add_subplot(2, 5, r * 5 + c + 1, projection=projection) if HAS_CARTOPY else fig.add_subplot(2, 5, r * 5 + c + 1)
                axes.append(ax)
                if HAS_CARTOPY:
                    ax.add_feature(cfeature.LAND, facecolor="#d8d8d8", edgecolor="none", zorder=0)
                    ax.coastlines(resolution="50m", linewidth=0.35, color="0.35")
                    ax.set_extent([-180, 180, 0, 90], crs=ccrs.PlateCarree())
                    trans = ccrs.PlateCarree()
                else:
                    trans = None
                imp = ims_map[(pid, spec["key"])]
                valid = domain & np.isfinite(imp)
                if HAS_CARTOPY:
                    ax.scatter(lon[domain], lat[domain], s=0.35, c="#cfcfcf", marker="s", linewidths=0, alpha=0.7, transform=trans, rasterized=True, zorder=1)
                    scatter = ax.scatter(lon[valid], lat[valid], c=imp[valid], s=0.35, cmap=cmap, norm=norm, marker="s", linewidths=0, transform=trans, rasterized=True, zorder=2)
                else:
                    scatter = ax.scatter(lon[valid], lat[valid], c=imp[valid], s=0.35, cmap=cmap, norm=norm)
                row = ims_summary[(ims_summary.period_id == pid) & (ims_summary.metric == spec["key"])].iloc[0]
                if r == 0:
                    ax.set_title(spec["label"], fontsize=9.5)
                if c == 0:
                    ax.text(-0.23, 0.5, f"{pid}\n{P1P2_BY_ID.loc[pid, 'label']}", transform=ax.transAxes, ha="right", va="center", fontsize=8.3, fontweight="bold")
                panel_label(ax, f"({chr(97 + r * len(ims_metric_specs) + c)})", x=-0.055)
                ax.text(0.02, 0.03, f"mean {row.mean_improvement:+.3f}\nimproved {row.percent_cells_improved:.0f}%", transform=ax.transAxes, ha="left", va="bottom", fontsize=7, bbox={"facecolor":"white", "edgecolor":"none", "alpha":0.84, "pad":1.4})
    cbar = fig.colorbar(scatter, ax=axes, orientation="horizontal", fraction=0.05, pad=0.08)
    cbar.set_label("DA categorical skill improvement (fraction; red = improvement)")
    fig.suptitle("IMS snow-cover categorical skill improvement by MODIS SCF period", fontsize=12)
    savefig(fig, "p1_p2_ims_skill_maps")
    show_figure(fig)

## SNOTEL and GHCN Station Metrics

Station support is common across P1 and P2 and across OL and DA. SNOTEL SWE is primary; SNOTEL snow depth is retained as a secondary check. GHCN uses snow depth.

In [ ]:
def compute_station_period_metrics(ds_path, dataset, obs_var, model_var, variable_label, units, min_days=100):
    ds = xr.open_dataset(ds_path)
    rows = []
    common_stations = None
    station_ids = ds["station"].values.astype(str)
    lat_name = "station_lat"
    lon_name = "station_lon"
    station_lat = np.asarray(ds[lat_name].values, dtype=float)
    station_lon = np.asarray(ds[lon_name].values, dtype=float)

    per_period_metrics = {}
    for pid, p in P1P2_BY_ID.iterrows():
        sub = ds.sel(time=slice(str(p["start"].date()), str(p["end"].date())))
        obs = sub[obs_var]
        support_this = np.ones(sub.sizes["station"], dtype=bool)
        for exp in ["OL", "DA"]:
            met = vector_metrics(obs, sub[model_var].sel(exp=exp), dim="time", min_pairs=min_days).load()
            per_period_metrics[(pid, exp)] = met
            support_this &= np.asarray(met["N"].values >= min_days)
        common_stations = support_this if common_stations is None else (common_stations & support_this)

    for pid, p in P1P2_BY_ID.iterrows():
        for exp in ["OL", "DA"]:
            met = per_period_metrics[(pid, exp)]
            for i, station in enumerate(station_ids):
                if not common_stations[i]:
                    continue
                rec = {
                    "dataset": dataset,
                    "variable": variable_label,
                    "units": units,
                    "period_id": pid,
                    "period_label": p["label"],
                    "experiment": exp,
                    "station": station,
                    "station_lat": station_lat[i],
                    "station_lon": station_lon[i],
                }
                for key in ["N", "R", "bias", "abs_bias", "rmse", "ubrmse"]:
                    rec[key] = float(met[key].values[i])
                rows.append(rec)
    ds.close()
    out = pd.DataFrame(rows)
    return out


def station_delta_summary(station_metrics):
    rows = []
    for (dataset, variable, pid), sub in station_metrics.groupby(["dataset", "variable", "period_id"]):
        p = P1P2_BY_ID.loc[pid]
        for metric in ["R", "rmse", "ubrmse", "abs_bias"]:
            piv = sub.pivot(index="station", columns="experiment", values=metric).dropna()
            if piv.empty:
                continue
            delta = improvement_delta(metric, piv["DA"], piv["OL"])
            rows.append({
                "dataset": dataset,
                "variable": variable,
                "period_id": pid,
                "period_label": p["label"],
                "metric": metric,
                "improvement_definition": "DA - OL" if metric == "R" else "OL - DA",
                "n_stations_common": len(delta),
                "mean_improvement": float(np.nanmean(delta)),
                "median_improvement": float(np.nanmedian(delta)),
                "percent_stations_improved": float(np.nanmean(delta > 0) * 100.0),
                "ol_mean": float(np.nanmean(piv["OL"])),
                "da_mean": float(np.nanmean(piv["DA"])),
            })
    return pd.DataFrame(rows)

In [ ]:
station_metric_frames = []
if RUN_SNOTEL:
    snotel_swe = compute_station_period_metrics(
        INPUTS["snotel_raw"], "SNOTEL", "obs_swe_kgm2", "model_swe_kgm2", "SWE", "kg m-2", COMMON_SUPPORT["station_min_common_days"]
    )
    station_metric_frames.append(snotel_swe)
    snotel_snwd = compute_station_period_metrics(
        INPUTS["snotel_raw"], "SNOTEL", "obs_snwd_m", "model_snwd_m", "Snow depth", "m", COMMON_SUPPORT["station_min_common_days"]
    )
    station_metric_frames.append(snotel_snwd)
if RUN_GHCN:
    ghcn_snwd = compute_station_period_metrics(
        INPUTS["ghcn_raw"], "GHCN", "obs_snwd_mm", "model_snwd_mm", "Snow depth", "mm", COMMON_SUPPORT["station_min_common_days"]
    )
    station_metric_frames.append(ghcn_snwd)

if station_metric_frames:
    station_metrics = pd.concat(station_metric_frames, ignore_index=True)
    station_metrics.to_csv(OUT_DIR / "p1_p2_station_metrics_common_support.csv", index=False)
    station_summary = station_delta_summary(station_metrics)
    station_summary.to_csv(OUT_DIR / "p1_p2_station_delta_summary.csv", index=False)
    display(station_summary)

In [ ]:
if MAKE_MAPS and station_metric_frames:
    map_metric = "ubrmse"
    for dataset, variable in station_metrics[["dataset", "variable"]].drop_duplicates().itertuples(index=False):
        sub = station_metrics[(station_metrics.dataset == dataset) & (station_metrics.variable == variable)]
        fig = plt.figure(figsize=(10.8, 3.6 if dataset == "SNOTEL" else 3.2))
        projection = ccrs.LambertConformal(central_longitude=-105, central_latitude=45) if HAS_CARTOPY and dataset == "SNOTEL" else (ccrs.Robinson() if HAS_CARTOPY else None)
        scatter = None
        for i, pid in enumerate(["P1", "P2"]):
            ax = fig.add_subplot(1, 2, i + 1, projection=projection) if HAS_CARTOPY else fig.add_subplot(1, 2, i + 1)
            psub = sub[sub.period_id == pid]
            piv = psub.pivot(index="station", columns="experiment", values=map_metric).dropna()
            loc = psub.drop_duplicates("station").set_index("station").reindex(piv.index)
            delta = improvement_delta(map_metric, piv["DA"], piv["OL"])
            vmax = np.nanpercentile(np.abs(delta), 95) if len(delta) else 1
            vmax = max(vmax, 1e-6)
            norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
            if HAS_CARTOPY:
                extent = [-170, -60, 20, 75] if dataset == "SNOTEL" else [-180, 180, 20, 90]
                add_geo_base(ax, extent=extent)
                scatter = ax.scatter(loc.station_lon, loc.station_lat, c=delta, s=12 if dataset == "GHCN" else 22, cmap="RdBu_r", norm=norm, transform=ccrs.PlateCarree(), linewidths=0.15, edgecolors="0.2", zorder=2)
            else:
                scatter = ax.scatter(loc.station_lon, loc.station_lat, c=delta, s=12 if dataset == "GHCN" else 22, cmap="RdBu_r", norm=norm)
            ax.set_title(f"{pid}: {P1P2_BY_ID.loc[pid, 'label']}", fontsize=10)
            panel_label(ax, f"({chr(97 + i)})")
            ax.text(0.02, 0.03, f"mean {np.nanmean(delta):+.3g}\nimproved {np.nanmean(delta > 0) * 100:.0f}%", transform=ax.transAxes, ha="left", va="bottom", fontsize=8, bbox={"facecolor":"white", "edgecolor":"none", "alpha":0.85, "pad":2})
        cbar = fig.colorbar(scatter, ax=fig.axes, orientation="horizontal", fraction=0.07, pad=0.05)
        cbar.set_label(f"{map_metric} improvement (OL - DA; positive/red = DA better) [{sub.units.iloc[0]}]")
        fig.suptitle(f"{dataset} {variable}: P1 vs P2 station-map improvement on common support", fontsize=12)
        safe_var = variable.lower().replace(" ", "_")
        savefig(fig, f"p1_p2_{dataset.lower()}_{safe_var}_{map_metric}_station_maps")
        show_figure(fig)

## ERA5-Land Gridded Metrics

This block reads the aligned monthly ERA5-Land vs OL/DA fields. It is I/O heavy, so `RUN_ERA5L` is `False` by default. Common support is defined per variable/metric over P1, P2, OL, and DA.

In [ ]:
def era5l_metric_maps_for_variable(var_prefix, min_months_by_period=None):
    min_months_by_period = min_months_by_period or COMMON_SUPPORT["era5l_min_common_months"]
    var_model = f"{var_prefix}_model"
    var_era = f"{var_prefix}_era"
    out_maps = {}
    support = None
    for exp, path in [("OL", INPUTS["era5l_ol"]), ("DA", INPUTS["era5l_da"] )]:
        ds = xr.open_dataset(path)
        for pid, p in P1P2_BY_ID.iterrows():
            sub = ds.sel(time=slice(str(p["start"].date()), str(p["end"].date())))
            met = vector_metrics(sub[var_era], sub[var_model], dim="time", min_pairs=min_months_by_period[pid]).load()
            out_maps[(pid, exp)] = met
            valid = np.asarray(np.isfinite(met["rmse"].values))
            support = valid if support is None else (support & valid)
        ds.close()
    return out_maps, support


def summarize_era5l_maps(out_maps, support, dataset_label, var_label):
    rows = []
    for pid, p in P1P2_BY_ID.iterrows():
        for metric in ["R", "rmse", "ubrmse", "abs_bias"]:
            ol = out_maps[(pid, "OL")][metric].values
            da = out_maps[(pid, "DA")][metric].values
            delta = improvement_delta(metric, da, ol)
            valid = support & np.isfinite(delta)
            rows.append({
                "dataset": dataset_label,
                "variable": var_label,
                "period_id": pid,
                "period_label": p["label"],
                "metric": metric,
                "improvement_definition": "DA - OL" if metric == "R" else "OL - DA",
                "n_cells_common": int(valid.sum()),
                "mean_improvement": float(np.nanmean(delta[valid])),
                "median_improvement": float(np.nanmedian(delta[valid])),
                "percent_cells_improved": float(np.nanmean(delta[valid] > 0) * 100.0),
                "ol_mean": float(np.nanmean(ol[valid])),
                "da_mean": float(np.nanmean(da[valid])),
            })
    return pd.DataFrame(rows)

In [ ]:
if RUN_ERA5L:
    era5_specs = [
        ("SM", "Surface soil moisture", "m3 m-3"),
        ("RZ", "Root-zone soil moisture", "m3 m-3"),
        ("SCF", "Snow-cover fraction", "fraction"),
        ("SWE", "SWE", "m"),
        ("SNWD", "Snow depth", "m"),
    ]
    era5_summary_frames = []
    era5_maps = {}
    era5_support = {}
    for var_prefix, var_label, units in era5_specs:
        maps, support = era5l_metric_maps_for_variable(var_prefix)
        era5_maps[var_prefix] = maps
        era5_support[var_prefix] = support
        era5_summary_frames.append(summarize_era5l_maps(maps, support, "ERA5-Land", var_label))
    era5_summary = pd.concat(era5_summary_frames, ignore_index=True)
    era5_summary.to_csv(OUT_DIR / "p1_p2_era5land_delta_summary.csv", index=False)
    display(era5_summary)

In [ ]:
if RUN_ERA5L and MAKE_MAPS:
    # Focused maps: soil-moisture ubRMSE and snow RMSE improvements.
    era5_map_specs = [("SM", "ubrmse", "Surface SM ubRMSE"), ("RZ", "ubrmse", "Root-zone SM ubRMSE"), ("SCF", "rmse", "SCF RMSE"), ("SWE", "rmse", "SWE RMSE"), ("SNWD", "rmse", "Snow-depth RMSE")]
    with xr.open_dataset(INPUTS["era5l_ol"]) as template:
        lats = np.asarray(template["lat"].values, dtype=float)
        lons = np.asarray(template["lon"].values, dtype=float)
    if lats.ndim == 1 and lons.ndim == 1:
        lon2d, lat2d = np.meshgrid(lons, lats)
    else:
        lon2d, lat2d = np.broadcast_arrays(lons, lats)
    coord_valid = np.isfinite(lon2d) & np.isfinite(lat2d)
    for var_prefix, metric, title in era5_map_specs:
        fig = plt.figure(figsize=(11.5, 3.6))
        projection = ccrs.Robinson() if HAS_CARTOPY else None
        scatter = None
        ims = []
        for i, pid in enumerate(["P1", "P2"]):
            ax = fig.add_subplot(1, 2, i + 1, projection=projection) if HAS_CARTOPY else fig.add_subplot(1, 2, i + 1)
            ol = era5_maps[var_prefix][(pid, "OL")][metric].values
            da = era5_maps[var_prefix][(pid, "DA")][metric].values
            delta = improvement_delta(metric, da, ol)
            delta = np.where(era5_support[var_prefix], delta, np.nan)
            vmax = np.nanpercentile(np.abs(delta), 98)
            vmax = max(float(vmax), 1e-8)
            norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)
            if HAS_CARTOPY:
                add_geo_base(ax)
                plot_valid = coord_valid & np.isfinite(delta)
                im = ax.scatter(lon2d[plot_valid], lat2d[plot_valid], c=delta[plot_valid], s=1.0, marker="s", transform=ccrs.PlateCarree(), cmap="RdBu_r", norm=norm, linewidths=0, rasterized=True)
            else:
                im = ax.imshow(delta, cmap="RdBu_r", norm=norm, origin="lower")
            ims.append(im)
            ax.set_title(f"{pid}: {P1P2_BY_ID.loc[pid, 'label']}", fontsize=10)
            panel_label(ax, f"({chr(97 + i)})")
            valid = np.isfinite(delta)
            ax.text(0.02, 0.03, f"mean {np.nanmean(delta):+.3g}\nimproved {np.nanmean(delta[valid] > 0) * 100:.0f}%", transform=ax.transAxes, ha="left", va="bottom", fontsize=8, bbox={"facecolor":"white", "edgecolor":"none", "alpha":0.85, "pad":2})
        cbar = fig.colorbar(ims[0], ax=fig.axes, orientation="horizontal", fraction=0.07, pad=0.05)
        cbar.set_label(f"{title} improvement (positive/red = DA better)")
        fig.suptitle(f"ERA5-Land {title}: P1 vs P2 common-cell maps", fontsize=12)
        savefig(fig, f"p1_p2_era5land_{var_prefix.lower()}_{metric}_maps")
        show_figure(fig)

## Cross-Dataset Rollup

Combines the summary tables that have been generated in this notebook. This cell can be rerun after enabling heavier sections.

In [ ]:
summary_files = [
    OUT_DIR / "p1_p2_ofa_summary.csv",
    OUT_DIR / "p1_p2_ims_summary.csv",
    OUT_DIR / "p1_p2_station_delta_summary.csv",
    OUT_DIR / "p1_p2_era5land_delta_summary.csv",
]
rollup_rows = []
for path in summary_files:
    if not path.exists():
        continue
    df = pd.read_csv(path)
    source = path.name
    if source == "p1_p2_ofa_summary.csv":
        keep_counts = df[(df["metric"] == "obs_per_day_common_tiles") & (df["experiment"] == "DA")]
        keep_delta = df[(df["metric"] == "spatial_mean_OmF_stdv_improvement_common_tiles") & (df["experiment"] == "DA_vs_OL")]
        keep = pd.concat([keep_counts, keep_delta], ignore_index=True)
        for _, r in keep.iterrows():
            rollup_rows.append({"source": source, "dataset": "OFA", "variable": r["group"], "period_id": r["period_id"], "metric": r["metric"], "value": r["value"], "unit": r["unit"], "n_support": r["n_tiles_common"]})
    elif source == "p1_p2_ims_summary.csv":
        keep = df[df["metric"].isin(["accuracy", "hit_rate", "false_alarm_ratio", "correct_rejection_rate"])]
        for _, r in keep.iterrows():
            rollup_rows.append({"source": source, "dataset": "IMS", "variable": "Snow cover", "period_id": r["period_id"], "metric": f"{r['metric']}_mean_improvement", "value": r["mean_improvement"], "unit": "fraction", "n_support": r["n_cells"]})
    elif source == "p1_p2_station_delta_summary.csv":
        for _, r in df.iterrows():
            rollup_rows.append({"source": source, "dataset": r["dataset"], "variable": r["variable"], "period_id": r["period_id"], "metric": f"{r['metric']}_mean_improvement", "value": r["mean_improvement"], "unit": "metric units", "n_support": r["n_stations_common"]})
    elif source == "p1_p2_era5land_delta_summary.csv":
        for _, r in df.iterrows():
            rollup_rows.append({"source": source, "dataset": r["dataset"], "variable": r["variable"], "period_id": r["period_id"], "metric": f"{r['metric']}_mean_improvement", "value": r["mean_improvement"], "unit": "metric units", "n_support": r["n_cells_common"]})

cross_dataset_rollup = pd.DataFrame(rollup_rows)
if len(cross_dataset_rollup):
    cross_dataset_rollup.to_csv(OUT_DIR / "p1_p2_cross_dataset_rollup.csv", index=False)
    display(cross_dataset_rollup)
else:
    print("No summary files found yet. Run sections above first.")